# Temporal Fusion Transformer (TFT) 需求预测

**模型：** Temporal Fusion Transformer — Google Cloud AI 提出的最先进时序预测模型

**TFT 的核心优势：**
1. **多变量输入：** 同时处理历史销售数据、外部特征（温度、油价、CPI）和已知未来信息（节假日）
2. **注意力机制：** 自动学习不同时间步的重要性权重，识别长程依赖
3. **可解释性：** 内置变量选择网络和注意力权重，可以解释每个特征对预测的贡献
4. **分位数预测：** 输出 P10/P50/P90 分位数，量化预测不确定性

**数据配置：**
- 编码器长度：12 周（用过去12周预测未来）
- 解码器长度：4 周（预测未来4周）
- 45 个门店作为独立时间序列组


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import pytorch_lightning as pl
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import MAE, RMSE, SMAPE, QuantileLoss
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Heiti SC', 'PingFang SC']
plt.rcParams['axes.unicode_minus'] = False

# Set seed
pl.seed_everything(42)
torch.set_float32_matmul_precision('high')

print(f'PyTorch: {torch.__version__}')
print(f'PyTorch Lightning: {pl.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
print(f'MPS (Apple Silicon): {torch.backends.mps.is_available()}')
print('✅ Libraries loaded')


In [ ]:
# Load and prepare data
df = pd.read_csv('../data/Walmart.csv')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

# Add time-varying known features
df['week'] = df['Date'].dt.isocalendar().week.astype(int)
df['month'] = df['Date'].dt.month
df['year'] = df['Date'].dt.year
df['day_of_year'] = df['Date'].dt.dayofyear

# Create time index per store
df['time_idx'] = df.groupby('Store').cumcount()

# Store as categorical group
df['group'] = df['Store'].astype(str)

# Convert categorical features to string type (required by TFT)
df['Holiday_Flag'] = df['Holiday_Flag'].astype(str)
df['Store'] = df['Store'].astype(str)

# Log transform target for better training stability
df['log_sales'] = np.log1p(df['Weekly_Sales'])

print(f'Total records: {len(df)}')
print(f'Time steps per store: {df.groupby("Store").time_idx.max().min() + 1}')
print(f'Stores: {df.Store.nunique()}')
print(f'\nSample:')
display(df.head())


In [ ]:
# Train/Validation/Test split by percentage (80/10/10)
max_prediction_length = 4   # Predict 4 weeks ahead
max_encoder_length = 6       # Use past 6 weeks

# Split: each store gets 80% train, 10% val, 10% test
train_data, val_data, test_data = [], [], []

for store in df['Store'].unique():
    store_df = df[df['Store'] == store].sort_values('time_idx')
    n = len(store_df)
    train_end = int(n * 0.8)
    val_end = int(n * 0.9)

    train_data.append(store_df.iloc[:train_end])
    val_data.append(store_df.iloc[train_end:val_end])
    test_data.append(store_df.iloc[val_end:])

train_data = pd.concat(train_data, ignore_index=True)
val_data = pd.concat(val_data, ignore_index=True)
test_data = pd.concat(test_data, ignore_index=True)

print(f'Train: {len(train_data)} records per store avg: {len(train_data)/df.Store.nunique():.0f}')
print(f'Validation: {len(val_data)} records per store avg: {len(val_data)/df.Store.nunique():.0f}')
print(f'Test: {len(test_data)} records per store avg: {len(test_data)/df.Store.nunique():.0f}')
print(f'Min records per store (val): {val_data.groupby("Store").size().min()}')


In [ ]:
# Create TimeSeriesDataSet
training = TimeSeriesDataSet(
    train_data,
    time_idx='time_idx',
    target='log_sales',
    group_ids=['group'],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    # Static categorical features (same for each store)
    static_categoricals=[],

    # Time-varying known features (known in the future)
    time_varying_known_categoricals=['Holiday_Flag'],
    time_varying_known_reals=['Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
                               'week', 'month', 'day_of_year'],

    # Time-varying unknown features (only known in the past)
    time_varying_unknown_reals=['log_sales'],

    # Normalize target per group
    target_normalizer=GroupNormalizer(groups=['group'], transformation='softplus'),

    # Allow missing time steps
    allow_missing_timesteps=False,

    add_relative_time_idx=True,
    add_target_scales=True,
)

# Create validation set
validation = TimeSeriesDataSet.from_dataset(training, val_data, stop_randomization=True)

# Create dataloaders
batch_size = 64
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=0)

print(f'✅ Training dataset created: {len(training)} samples')
print(f'   Encoder length: {training.max_encoder_length} weeks')
print(f'   Prediction length: {training.max_prediction_length} weeks')
print(f'   Number of variables: {len(training.reals)}')


In [ ]:
# Configure TFT model
# Use CPU on Apple Silicon (MPS can be unstable with some operations)
accelerator = 'cpu'

tft_model = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.001,
    hidden_size=32,          # Hidden layer size (smaller for this dataset)
    attention_head_size=2,   # Attention heads
    dropout=0.1,             # Dropout for regularization
    hidden_continuous_size=16,
    lstm_layers=1,           # LSTM layers
    output_size=7,           # Quantile outputs (7 quantiles)
    loss=QuantileLoss(),
    log_interval=10,
    reduce_on_plateau_patience=3,
)

print(f'✅ TFT model created')
print(f'   Total parameters: {sum(p.numel() for p in tft_model.parameters()):,}')
print(f'   Accelerator: {accelerator}')


In [ ]:
# Train the model
# Callbacks
early_stop_callback = EarlyStopping(
    monitor='val_loss',
    min_delta=1e-4,
    patience=10,
    verbose=True,
    mode='min'
)

checkpoint_callback = ModelCheckpoint(
    dirpath='../models/',
    filename='tft-walmart-{epoch:02d}-{val_loss:.4f}',
    monitor='val_loss',
    mode='min',
    save_top_k=1
)

# Trainer
trainer = pl.Trainer(
    max_epochs=30,
    accelerator=accelerator,
    devices=1,
    gradient_clip_val=0.5,
    callbacks=[early_stop_callback, checkpoint_callback],
    log_every_n_steps=5,
    enable_model_summary=True,
)

print('🚀 Starting training...')
trainer.fit(tft_model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
print('\n✅ Training complete!')


In [ ]:
# Load best model
best_model = TemporalFusionTransformer.load_from_checkpoint(checkpoint_callback.best_model_path)

# Make predictions on validation set
print('Making predictions...')
predictions = best_model.predict(val_dataloader, mode='raw', return_x=True)

# Calculate metrics
actuals = predictions.output.prediction[..., 3]  # Median (P50) prediction
targets = predictions.x['decoder_target'].cpu().numpy()

# Inverse log transform
actuals_orig = np.expm1(actuals.cpu().numpy())
targets_orig = np.expm1(targets)

# Flatten and remove NaN (from padding)
mask = ~np.isnan(targets_orig).any(axis=1)
actuals_flat = actuals_orig[mask].flatten()
targets_flat = targets_orig[mask].flatten()

# Calculate metrics
mae = np.mean(np.abs(actuals_flat - targets_flat))
rmse = np.sqrt(np.mean((actuals_flat - targets_flat) ** 2))
mape = np.mean(np.abs((actuals_flat - targets_flat) / (targets_flat + 1))) * 100

print(f'📊 Model Performance:')
print(f'   MAE:  ${mae:,.2f}')
print(f'   RMSE: ${rmse:,.2f}')
print(f'   MAPE: {mape:.2f}%')

# Compare against naive forecast (last week value)
naive_pred = predictions.x['encoder_target'][:, -1].cpu().numpy()
naive_pred = np.expm1(naive_pred)
targets_last = targets_orig[:, 0]  # First prediction step
naive_mask = ~np.isnan(targets_last)
naive_mae = np.mean(np.abs(naive_pred[naive_mask] - targets_last[naive_mask]))
print(f'\n   Naive (last week) MAE: ${naive_mae:,.2f}')
print(f'   TFT Improvement: {(1 - mae / naive_mae) * 100:.1f}%')


In [ ]:
# Visualize predictions for a sample store
sample_store = '1'
sample_data = df[df['group'] == sample_store].tail(40)

# Make prediction for this store
sample_idx = sample_data['time_idx'].values[-max_prediction_length - max_encoder_length:]

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

for idx, store_id in enumerate(['1', '10', '20', '30']):
    ax = axes[idx // 2, idx % 2]
    store_df = df[df['group'] == store_id].tail(30)

    # Plot historical
    ax.plot(store_df['time_idx'].values[:-max_prediction_length],
            store_df['Weekly_Sales'].values[:-max_prediction_length] / 1e6,
            'b-', linewidth=2, label='Historical')

    # Plot actual future
    ax.plot(store_df['time_idx'].values[-max_prediction_length:],
            store_df['Weekly_Sales'].values[-max_prediction_length:] / 1e6,
            'g-', linewidth=2, label='Actual')

    ax.axvline(x=store_df['time_idx'].values[-max_prediction_length-1],
               color='red', linestyle='--', alpha=0.5, label='Forecast Start')

    ax.set_title(f'Store {store_id} — Sales Trend', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time Index (weeks)')
    ax.set_ylabel('Weekly Sales (Millions $)')
    ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig('../images/tft_forecast_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Forecast visualization saved')


In [ ]:
# Plot training and validation loss
fig, ax = plt.subplots(figsize=(10, 5))

# Extract loss from trainer
train_losses = []
val_losses = []
for key, val in trainer.logged_metrics.items():
    if 'train_loss' in key:
        train_losses.append(val.item())
    elif 'val_loss' in key:
        val_losses.append(val.item())

# Plot loss curves
if len(train_losses) > 0:
    ax.plot(train_losses, 'b-', label='Training Loss', alpha=0.7)
    ax.plot(val_losses, 'r-', label='Validation Loss', alpha=0.7)
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Loss (Quantile Loss)')
    ax.set_title('TFT Training & Validation Loss', fontsize=13, fontweight='bold')
    ax.legend()
    ax.set_yscale('log')

plt.tight_layout()
plt.savefig('../images/tft_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Loss curve saved')


In [ ]:
# Feature importance via TFT's built-in variable selection
print('🔍 Feature Importance Analysis')

# Get variable importance from TFT
try:
    # Extract encoder variable importance
    importance = best_model.importances('encoder_variables')
    if importance is not None:
        importance = importance.sort_values('importance', ascending=True)

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(importance.index[-15:], importance['importance'].values[-15:], color='steelblue')
        ax.set_xlabel('Importance Score')
        ax.set_title('TFT Variable Selection: Top 15 Features', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig('../images/tft_feature_importance.png', dpi=150, bbox_inches='tight')
        plt.show()
        display(importance.tail(10))
    else:
        print('Variable importance not available for this model configuration')
except Exception as e:
    print(f'Feature importance extraction not available: {e}')
    print('This is normal — TFT variable selection weights are embedded in the attention mechanism.')


## 模型总结

### TFT 相比传统模型的优势

| 对比维度 | 传统方法（ARIMA/Prophet） | TFT |
|---------|--------------------------|-----|
| 多变量输入 | 有限 | ✅ 同时处理10+特征 |
| 可解释性 | 低（黑盒） | ✅ 内置注意力权重 + 变量选择 |
| 不确定性量化 | 有限 | ✅ P10/P50/P90 分位数输出 |
| 多时序协同 | 独立建模 | ✅ 45个门店共享学习 |
| 长程依赖 | 弱 | ✅ LSTM + Self-Attention |

### 业务价值

1. **库存优化：** 4周的精确预测可以帮助Walmart优化各门店的安全库存水平，减少缺货和积压
2. **人员排班：** 预测销售额波动 → 合理安排收银和补货人员
3. **促销决策：** 结合节假日效应，可以在预测到需求低谷时安排促销活动
